# Raster Data Format

**Raster spatial data** is a way of representing geographic information as a regular grid (matrix) of equal-sized cells — pixels. Each cell covers a specific area on the ground and holds a numeric value (or a set of values), such as elevation, temperature, reflected radiation intensity, and so on.

The raster format is especially well suited for data that **varies continuously across space** — that is, for surfaces such as terrain, precipitation, temperature, soil moisture, and similar phenomena.

Pixel size determines the **spatial resolution** of the data: the smaller the pixel, the more detail the raster captures.

We had a brief introduction to the raster data format in the [first section](../module_1/spData_1.ipynb).

In this section we will work with real population data from the [WorldPop](https://hub.worldpop.org) portal: a raster in which every pixel holds the estimated **number of people** living in it.

## 0. Importing Libraries

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling

import numpy as np
import matplotlib.pyplot as plt

import osmnx as ox

# keep all downloaded OSM responses in one cache folder at the root of the repository
ox.settings.cache_folder = "../../cache"

## 1. WorldPop Data

The population data on the [WorldPop](https://hub.worldpop.org) portal represents an estimate of population distribution as a raster surface, where each pixel holds a population count. These estimates are produced by combining census data with various auxiliary sources — for example, satellite imagery and data on built-up areas and infrastructure.

As a result, population is "distributed" across raster cells based on territorial characteristics: where the probability of people living is higher, the values are larger.

### 1.1. Downloading the Data

We'll download WorldPop population distribution data for 2020. The code below shows how to fetch the raster file from the official server. Note that the file is quite large (~300 MB), so the download may take some time.

If you prefer to skip this step, the file is already included in the course materials directory.

In the examples below we will use a pre-clipped file covering the Tula Oblast territory.

In [ ]:
# You can download the data from the WorldPop portal (~300 MB)

# import requests

# # WorldPop data URL
# url = "https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/RUS/rus_ppp_2020_constrained.tif"

# # Output filename
# output_file = "worldpop_russia_2020.tif"

# # Download the file
# response = requests.get(url, stream=True)

# if response.status_code == 200:
#     with open(output_file, "wb") as file:
#         for chunk in response.iter_content(chunk_size=1024):
#             file.write(chunk)
#     print(f"File saved as {output_file}")
# else:
#     print("Failed to download file. Status code:", response.status_code)

### 1.2. Reading Raster Data

Let's open the pre-clipped raster covering the Tula Oblast boundary using the `rasterio` library. The file **tula_region_population.tif** is available in the [repository](https://github.com/bella-mir/geoPythonEn/tree/main/data/tula).

Pixel values represent the estimated population count within each cell.

After opening the file, we'll print its key properties:

- **CRS (coordinate reference system)** — specifies the spatial reference system the data is defined in;
- **bounds** — the spatial extent of the raster;
- **resolution** — the size of one pixel in the units of the coordinate system.

In [ ]:
raster_path = "../../data/tula/tula_region_population.tif"

with rasterio.open(raster_path) as dataset:
    print(f"CRS: {dataset.crs}")
    print(f"Bounds: {dataset.bounds}")
    print(f"Resolution: {dataset.res}")

### 1.3. Visualising the Raster

Once the data is loaded, we can visualise it. A raster layer is essentially a matrix of values, so it can be displayed as an image using `matplotlib`.

In the example below we plot the first (and only) band of the raster and add a colorbar to help interpret the values.

In [ ]:
with rasterio.open(raster_path) as dataset:
    data = dataset.read(1)

plt.figure(figsize=(10, 10))
plt.imshow(data, cmap="viridis")
plt.colorbar(label="Population (people per pixel)")
plt.title("Population Distribution — Tula Oblast (WorldPop, 2020)")
plt.show()

- `dataset.read(1)`  
  reads the first band of the raster as a two-dimensional array (matrix);

- `plt.imshow(data, cmap="viridis")`  
  displays the matrix as an image, mapping pixel values to colours;

- `cmap="viridis"`  
  sets the colour scheme;

- `plt.colorbar()`  
  adds a scale bar showing which values correspond to which colours;

- `plt.title()`  
  adds a title for easier interpretation.

### 1.4. Handling NoData Values

In the previous step, the raster visualisation may have looked "distorted": a large portion of the territory appears in a very narrow colour range, and the colorbar is skewed.

This happens because the data contains a large number of strongly negative values that throw off the automatic colour scaling.

Let's find out what these values are by inspecting the basic statistics of the raster:

In [ ]:
with rasterio.open(raster_path) as dataset:
    data = dataset.read(1)
    # Print statistics
    print(f"Min value: {np.min(data)}")
    print(f"Max value: {np.max(data)}")
    print(f"Mean value: {np.mean(data)}")

We will see that the minimum raster value is `-99999`. This clearly cannot be a valid population count.

Such extreme (often negative) values in raster data are commonly used as **NoData values**, indicating the absence of information in a pixel (for example, outside the study area boundary).

To work with the data correctly, these pixels must be excluded from analysis.

#### 1.4.1. Masking NoData Values

`rasterio` can handle missing values automatically using a _masked array_:

In [ ]:
with rasterio.open(raster_path) as dataset:
    data = dataset.read(1, masked=True)

All pixels flagged as NoData are automatically masked and excluded from computations and visualisation.

Let's check the new minimum value of the raster:

In [ ]:
print(f"Min (masked): {data.min()}")

#### 1.4.2. Visualisation Without NoData

Now let's look at the raster with missing values excluded. The colour scale is no longer stretched by the `-99999` pixels, so the actual distribution of population becomes visible:

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(data, cmap="viridis")
plt.colorbar(label="Population (people per pixel)")
plt.title("Population Distribution — Tula Oblast (WorldPop, 2020)")
plt.show()

## 2. Clipping the Raster to a Vector Layer

In the previous steps we worked with a raster covering the entire Tula Oblast. In practice, however, you often need to analyse data for a specific area — for example, a particular city.

In such cases, raster data is **clipped** to the boundary of the area of interest.

As an example, we'll clip the data to the city of Tula.

To do this, we need the city boundary as a vector layer. We'll load it from OpenStreetMap using the `osmnx` library.

### 2.1. Loading Vector Data

Load the Tula city boundary from OpenStreetMap

In [ ]:
area_name = "Tula, Russia"

admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

### 2.2. Clipping the Raster

Now we'll clip the raster to the Tula city boundary.

We'll use the `mask` function from `rasterio`, which clips raster data to an arbitrary geometry.

Before clipping, it's important to make sure that the raster and the vector boundary share the **same coordinate reference system (CRS)** — otherwise the result will be incorrect.

#### 2.2.1. Checking Coordinate Reference Systems

In [ ]:
with rasterio.open(raster_path) as dataset:
    raster_crs = dataset.crs

print(f"Raster CRS: {raster_crs}")
print(f"Boundary CRS: {admin_border.crs}")

# If the CRS differ — reproject the boundary layer
if admin_border.crs != raster_crs:
    admin_border = admin_border.to_crs(raster_crs)

print(f"Same CRS: {admin_border.crs == raster_crs}")

#### 2.2.2. Preparing the Geometry

The `mask` function accepts any geometry object that shapely provides, so we can pass the boundary geometry straight from the `GeoDataFrame`:

In [ ]:
geometries = admin_border.geometry

#### 2.2.3. Clipping the Raster

In [ ]:
with rasterio.open(raster_path) as dataset:
    out_image, out_transform = mask(dataset, geometries, crop=True)

    # copy the metadata while the file is still open
    out_meta = dataset.meta.copy()

- `out_image` — the clipped raster array
- `out_transform` — the updated spatial transform

The `crop=True` parameter trims the raster extent to the bounding box of the geometry.

#### 2.2.4. Updating Metadata

The metadata was copied in the cell above, while the file was still open. It still describes the original raster, so we update the fields that clipping has changed:

- `driver` — the output format (`GTiff` for GeoTIFF);
- `height`, `width` — the dimensions of the clipped array, taken from `out_image.shape`;
- `transform` — the georeferencing of the clipped raster, returned by `mask()`.

In [ ]:
out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

#### 2.2.5. Saving the Result

We open a new file in write mode (`"w"`), pass the updated metadata as keyword arguments, and write the clipped array into it:

In [ ]:
cropped_path = "../../data/tula/tula_cropped_population.tif"

with rasterio.open(cropped_path, "w", **out_meta) as dest:
    dest.write(out_image)

### 2.3. Inspecting the Clipped Raster

After clipping, it's useful to check the output: review its key properties, pixel size, and visually confirm that the data is indeed limited to the Tula city boundary.

#### 2.3.1. Basic Raster Information

In [ ]:
with rasterio.open(cropped_path) as dataset:
    print(f"CRS: {dataset.crs}")
    print(f"Bounds: {dataset.bounds}")
    print(f"Width, Height: {dataset.width}, {dataset.height}")
    print(f"Number of bands: {dataset.count}")
    print(f"Data type: {dataset.dtypes}")
    print(f"Transform: {dataset.transform}")

Here we check:

- `CRS` — the raster's coordinate reference system;
- `Bounds` — the spatial extent;
- `Width, Height` — the raster dimensions in pixels;
- `Number of bands` — the number of bands;
- `Data type` — the pixel value type;
- `Transform` — the raster's georeferencing parameters.

Note that the raster is in a geographic coordinate system — **EPSG:4326**.

This means that:

- coordinates are expressed in degrees;
- pixel size is also in degrees, not metres.

This is important to keep in mind for further analysis, such as calculating distances, areas, or density.

#### 2.3.2. Pixel Size

The pixel size can be retrieved directly using the `res` attribute of `rasterio`:

In [ ]:
with rasterio.open(cropped_path) as dataset:
    pixel_width, pixel_height = dataset.res

    print(f"Pixel Width: {pixel_width}")
    print(f"Pixel Height: {pixel_height}")

- `pixel_width` — pixel size along the X axis;
- `pixel_height` — pixel size along the Y axis.

Keep in mind that in this case pixel size is expressed in **degrees**, since the raster is in the EPSG:4326 coordinate system.

#### 2.3.3. Visualising the Clipped Raster

Let's see what the raster layer looks like after clipping:

In [ ]:
with rasterio.open(cropped_path) as dataset:
    data = dataset.read(1, masked=True)

plt.figure(figsize=(10, 10))
plt.imshow(data, cmap="viridis")
plt.colorbar(label="Population (people per pixel)")
plt.title("Population Distribution — Tula City (WorldPop, 2020)")
plt.show()

## 3. Reprojecting the Raster

Our raster is currently in a geographic coordinate system (EPSG:4326), where coordinates and pixel size are expressed in degrees.

For most analytical tasks — calculating distances, areas, and density — this is inconvenient, so rasters are typically reprojected into a projected coordinate system with metric units, such as UTM.

### 3.1. Determining the Appropriate UTM Zone

The UTM zone depends on the geographic location of the study area. It can be determined automatically from the vector boundary layer, using the same `estimate_utm_crs()` method we used in the [second module](../module_2/projections_2.ipynb) — `osmnx` returns the boundary as a `GeoDataFrame`, so the method is available straight away:

In [ ]:
utm_crs = admin_border.estimate_utm_crs()
print(f"Estimated UTM CRS: {utm_crs}")

### 3.2. Reprojecting the Raster

We use functions from the `rasterio` library to perform the reprojection.

> When a raster has multiple bands, reprojection is applied to each band separately

**Choosing a resampling method.** Reprojection puts the data on a new pixel grid, so the values of the old pixels have to be transferred to the new ones somehow, and the method matters:

- `nearest` — copies the value of the closest source pixel. The usual choice for categorical rasters (land cover, classifications), where averaging values would be meaningless.
- `bilinear`, `cubic` — interpolate between neighbouring pixels. Suitable for continuous surfaces such as elevation or temperature.
- `sum` — adds up the values falling into each new pixel. This is what we need here: every WorldPop pixel stores a **population count**, and the total number of people must stay the same after reprojection.

#### 3.2.1. Computing the Target Transform

First, we calculate how the raster will look in the new coordinate system:

In [ ]:
with rasterio.open(cropped_path) as dataset:
    transform, width, height = calculate_default_transform(
        dataset.crs, utm_crs, dataset.width, dataset.height, *dataset.bounds
    )

    # copy the metadata while the file is still open
    utm_meta = dataset.meta.copy()

- `transform` — the new affine transform
- `width`, `height` — the new raster dimensions in pixels

#### 3.2.2. Updating Metadata

The metadata was copied in the cell above; now we update the fields that change with the new coordinate system — the CRS itself, the transform, and the raster dimensions:

In [ ]:
utm_meta.update({
        "crs": utm_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

#### 3.2.3. Setting the Output File Path

Define the path for saving the reprojected raster:

In [ ]:
reprojected_path = "../../data/tula/tula_cropped_population_utm.tif"

#### 3.2.4. Reprojecting the Data

Now we perform the reprojection and write the result to a new file. Two files are open at the same time here — the source (`src`) and the destination (`dst`) — and the loop runs over the bands of the raster, which in our case is a single one:

In [ ]:
with rasterio.open(cropped_path) as src:
    with rasterio.open(reprojected_path, "w", **utm_meta) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=utm_crs,
                resampling=Resampling.sum
            )

> During reprojection, the coordinates of each pixel are transformed into the new coordinate system, so the raster changes its shape and pixel size. We use `sum` resampling so that the population total carries over unchanged — with `nearest`, the same reprojection loses around 11% of the population simply because the new pixels do not line up with the old ones.

### 3.3. Verifying the Result

Let's check the new coordinate system and pixel size — and confirm that the reprojection did not lose any population:

In [ ]:
with rasterio.open(cropped_path) as dataset:
    total_before = dataset.read(1, masked=True).sum()

with rasterio.open(reprojected_path) as dataset:
    total_after = dataset.read(1, masked=True).sum()

    print(f"CRS: {dataset.crs}")
    print(f"Resolution: {dataset.res}")

print(f"Population before reprojection: {round(float(total_before))}")
print(f"Population after reprojection:  {round(float(total_after))}")

After reprojection:

- the coordinate system is now projected (UTM);
- pixel size is now expressed in **metres**;
- the population total is unchanged, so the raster can be used for counts and densities;
- the raster is ready for further spatial analysis.

## Summary

In this section we explored the raster data format and went through the core steps of working with it, using WorldPop data as an example.

We:

- learned how to open and inspect raster files using `rasterio`;
- identified and handled missing values (_NoData_);
- clipped the raster to a study area boundary;
- reprojected the data.

The result is a prepared raster layer ready for further spatial analysis — for example, assessing population distribution, aggregating data, or computing accessibility metrics.